In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
# Funciones auxiliares para mostrar los cálculos intermedios paso a paso
from itertools import combinations as _combinations


def detalle_tiempos_nodos(proyecto, duraciones):
    """Tiempos tempranos y tardíos de los nodos con los candidatos que se comparan en cada uno."""
    G = proyecto.pert_graph
    d = {data['activity']: (0 if data['activity'].startswith('@') else int(duraciones[data['activity']]))
         for _, _, data in G.edges(data=True)}
    nodos = list(proyecto.nodes)
    early, filas = {}, []
    for j in nodos:
        entrantes = [(i, data['activity']) for i, _, data in G.in_edges(j, data=True)]
        if not entrantes:
            early[j] = 0
            filas.append({'nodo': j, 'candidatos t_i + d_ij': 'nodo inicial', 'tiempo temprano = máx': 0})
            continue
        cand = {f'{a}: {early[i]} + {d[a]} = {early[i] + d[a]}': early[i] + d[a] for i, a in entrantes}
        early[j] = max(cand.values())
        filas.append({'nodo': j, 'candidatos t_i + d_ij': '; '.join(cand), 'tiempo temprano = máx': early[j]})
    tempranos = pd.DataFrame(filas).set_index('nodo')
    late, filas = {}, []
    for i in reversed(nodos):
        salientes = [(j, data['activity']) for _, j, data in G.out_edges(i, data=True)]
        if not salientes:
            late[i] = early[i]
            filas.append({'nodo': i, 'candidatos t*_j − d_ij': 'nodo final (= su tiempo temprano)', 'tiempo tardío = mín': late[i]})
            continue
        cand = {f'{a}: {late[j]} − {d[a]} = {late[j] - d[a]}': late[j] - d[a] for j, a in salientes}
        late[i] = min(cand.values())
        filas.append({'nodo': i, 'candidatos t*_j − d_ij': '; '.join(cand), 'tiempo tardío = mín': late[i]})
    tardios = pd.DataFrame(filas).set_index('nodo')
    return tempranos, tardios


def detalle_holguras(calendario, prelaciones):
    """Cómo se obtiene cada fecha y cada holgura a partir de las vecinas."""
    pred = prelaciones.immediate_predecessor
    suc = {a: sorted(s for s, ps in pred.items() if a in ps) for a in calendario.index}
    fin = int(calendario['fin_mas_temprano'].max())
    filas = []
    for a in calendario.index:
        ES, EF = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'fin_mas_temprano'])
        LS, LF = int(calendario.loc[a, 'inicio_mas_tardio']), int(calendario.loc[a, 'fin_mas_tardio'])
        dur = int(calendario.loc[a, 'duracion'])
        ps, ss = sorted(pred[a]), suc[a]
        ef_pred = {p: int(calendario.loc[p, 'fin_mas_temprano']) for p in ps}
        es_suc = {s: int(calendario.loc[s, 'inicio_mas_temprano']) for s in ss}
        lf_pred = {p: int(calendario.loc[p, 'fin_mas_tardio']) for p in ps}
        ls_suc = {s: int(calendario.loc[s, 'inicio_mas_tardio']) for s in ss}
        min_es_suc = min(es_suc.values()) if es_suc else fin
        max_lf_pred = max(lf_pred.values()) if lf_pred else 0
        filas.append({
            'actividad': a,
            'ES = máx EF pred.': (f"máx({', '.join(f'{p}: {v}' for p, v in ef_pred.items())}) = {ES}" if ps else f'inicial → {ES}'),
            'EF = ES + d': f'{ES} + {dur} = {EF}',
            'LF = mín LS suc.': (f"mín({', '.join(f'{s}: {v}' for s, v in ls_suc.items())}) = {LF}" if ss else f'final → {LF}'),
            'LS = LF − d': f'{LF} − {dur} = {LS}',
            'H_total = LS − ES': f'{LS} − {ES} = {LS - ES}',
            'H_libre = mín ES suc. − EF': f"{min_es_suc}{'' if ss else ' (fin)'} − {EF} = {min_es_suc - EF}",
            'H_indep. = máx(0, mín ES suc. − máx LF pred. − d)': f'máx(0, {min_es_suc} − {max_lf_pred} − {dur}) = {max(0, min_es_suc - max_lf_pred - dur)}',
        })
    return pd.DataFrame(filas).set_index('actividad')


def detalle_rutas(R, valores, nombre='duración', decimales=0):
    """Suma término a término de un valor (duraciones, varianzas…) a lo largo de cada ruta."""
    filas = {}
    for r in R.index:
        acts = list(R.columns[R.loc[r] == 1])
        total = float(sum(valores[a] for a in acts))
        filas[r] = {'actividades': ' - '.join(acts),
                    f'{nombre} = suma': ' + '.join(f'{a}({valores[a]:.{decimales}f})' for a in acts) + f' = {total:.{decimales}f}',
                    nombre: total}
    return pd.DataFrame(filas).T


def detalle_estimaciones(datos, a='a', m='m', b='b'):
    """Media y varianza PERT de cada actividad con los números sustituidos."""
    filas = {}
    for act, fila in datos.iterrows():
        A, M, B = float(fila[a]), float(fila[m]), float(fila[b])
        filas[act] = {'D = (a + 4m + b)/6': f'({A:g} + 4·{M:g} + {B:g})/6 = {(A + 4 * M + B) / 6:.2f}',
                      'σ² = ((b − a)/6)²': f'(({B:g} − {A:g})/6)² = {((B - A) / 6) ** 2:.3f}'}
    return pd.DataFrame(filas).T


def detalle_ackoff(proyecto, duraciones, topes, cur, pasos):
    """En cada paso de Ackoff-Sasieni: rutas críticas, opciones consideradas (actividades comunes y parejas)
    con su coste, y la elegida."""
    R = proyecto.path_matrix(dummies=False)
    d = duraciones.astype(float).copy()
    filas = []
    for paso, elegidas in sorted(pasos.items()):
        L = R @ d[R.columns]
        T = float(L.max())
        criticas = [r for r in L.index if np.isclose(L[r], T)]
        conjuntos = [set(R.columns[R.loc[r] == 1]) for r in criticas]
        con_margen = {x for x in R.columns if d[x] > topes[x] and np.isfinite(cur[x])}
        opciones = {x: float(cur[x]) for x in set.intersection(*conjuntos) & con_margen}
        if len(conjuntos) > 1:
            cand = sorted(set().union(*conjuntos) & con_margen)
            for p, q in _combinations(cand, 2):
                if all(p in s or q in s for s in conjuntos) and p not in opciones and q not in opciones:
                    opciones[f'{p}+{q}'] = float(cur[p] + cur[q])
        orden = sorted(opciones.items(), key=lambda kv: kv[1])
        filas.append({'paso': paso + 1, 'duración': int(T), 'rutas críticas': ', '.join(criticas),
                      'opciones (coste)': '; '.join(f'{k} ({v:.0f})' for k, v in orden[:6]) + (' …' if len(orden) > 6 else ''),
                      'elegida': ', '.join(elegidas), 'coste del paso': float(sum(cur[x] for x in elegidas))})
        for x in elegidas:
            d[x] -= 1
    return pd.DataFrame(filas).set_index('paso')


def detalle_valor_ganado(datos_vg, calendario, semana):
    """PV, EV y AC actividad a actividad en el punto de control."""
    filas = {}
    for a, fila in datos_vg.iterrows():
        es, dur = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'duracion'])
        frac = min(max(semana - es, 0), dur) / dur
        filas[a] = {'BAC': float(fila['BAC']), 'planificado hasta la semana': f'{min(max(semana - es, 0), dur)} de {dur}',
                    'PV = BAC·fracción': f'{fila["BAC"]:.0f}·{frac:.2f} = {fila["BAC"] * frac:.1f}',
                    'realizado (%)': int(fila['performed']),
                    'EV = BAC·%': f'{fila["BAC"]:.0f}·{fila["performed"] / 100:.2f} = {fila["BAC"] * fila["performed"] / 100:.1f}',
                    'AC': float(fila['AC']), 'PV_i': fila['BAC'] * frac, 'EV_i': fila['BAC'] * fila['performed'] / 100}
    t = pd.DataFrame(filas).T
    return t


def muestra_formula(nombre, expresion, valor, unidades=''):
    """Una línea «nombre = expresión = valor» en Markdown."""
    return Markdown(f'**{nombre}** = {expresion} = **{valor}**{(" " + unidades) if unidades else ""}')

In [3]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Considere el proyecto cuyo cuadro de prelaciones y duraciones (semanas) se adjunta. Dos jefes de obra han propuesto sendas programaciones, dadas por la semana de inicio de cada actividad (segundo cuadro). Las actividades no se interrumpen.

In [4]:
pregunta_1 = "1. Para cada plan, compruebe actividad por actividad si respeta las prelaciones: una actividad no puede empezar antes de que terminen todas sus predecesoras. Liste las violaciones.  "
pregunta_2 = "2. Calcule la duración del proyecto que resulta de cada plan (fin de la última actividad) y compárela con la duración mínima del proyecto.  "
pregunta_3 = "3. Corrija cada plan con el mínimo cambio: adelante o retrase sólo las actividades necesarias (y sus sucesoras si hace falta) y recalcule la duración.  "
pregunta_4 = "4. ¿Qué plan es «más tardío» (deja menos margen) y qué riesgo supone? ¿Qué plan deja más holgura para absorber imprevistos?  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. Para cada plan, compruebe actividad por actividad si respeta las prelaciones: una actividad no puede empezar antes de que terminen todas sus predecesoras. Liste las violaciones.  

2. Calcule la duración del proyecto que resulta de cada plan (fin de la última actividad) y compárela con la duración mínima del proyecto.  

3. Corrija cada plan con el mínimo cambio: adelante o retrase sólo las actividades necesarias (y sus sucesoras si hace falta) y recalcule la duración.  

4. ¿Qué plan es «más tardío» (deja menos margen) y qué riesgo supone? ¿Qué plan deja más holgura para absorber imprevistos?  

In [5]:
#| tbl-cap: "Cuadro de datos del enunciado"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration
    ('A'     , '---'      , 7       ),
    ('B'     , '---'      , 5       ),
    ('C'     , '---'      , 4       ),
    ('D'     , 'A,B,C'    , 3       ),
    ('E'     , 'D'        , 1       ),
    ('F'     , 'E'        , 1       ),
    ('G'     , 'D'        , 7       ),
    ('H'     , 'F'        , 3       ),
], columns=['activity', 'predecessor', 'duration']).set_index('activity')
datos_enunciado

,predecessor,duration
activity,,
A,---,7
B,---,5
C,---,4
D,"A,B,C",3
E,D,1
F,E,1
G,D,7
H,F,3


El proyecto corresponde a la organización de un congreso científico; las actividades son las siguientes (las duraciones del cuadro están en semanas).

In [6]:
#| tbl-cap: "Descripción de las actividades"
#| label: tbl-descripciones
descripciones = pd.Series({
    'A': 'Definición del programa y fechas',
    'B': 'Reserva de la sede',
    'C': 'Convocatoria de comunicaciones',
    'D': 'Página web e inscripciones',
    'E': 'Contratación de ponentes invitados',
    'F': 'Revisión de los trabajos recibidos',
    'G': 'Elaboración del programa definitivo',
    'H': 'Contratación de catering y viajes',
}, name='Descripción')
descripciones.to_frame()

,Descripción
A,Definición del programa y fechas
B,Reserva de la sede
C,Convocatoria de comunicaciones
D,Página web e inscripciones
E,Contratación de ponentes invitados
F,Revisión de los trabajos recibidos
G,Elaboración del programa definitivo
H,Contratación de catering y viajes


In [7]:
#| tbl-cap: "Programaciones propuestas (semana de inicio)"
#| label: tbl-planes
planes = pd.DataFrame([
#  activity, start_plan_1, start_plan_2
    ('A'     , 0           , 0           ),
    ('B'     , 2           , 0           ),
    ('C'     , 3           , 0           ),
    ('D'     , 7           , 7           ),
    ('E'     , 9           , 12          ),
    ('F'     , 9           , 14          ),
    ('G'     , 10          , 10          ),
    ('H'     , 14          , 12          ),
], columns=['activity', 'start_plan_1', 'start_plan_2']).set_index('activity')
planes

,start_plan_1,start_plan_2
activity,,
A,0,0
B,2,0
C,3,0
D,7,7
E,9,12
F,9,14
G,10,10
H,14,12


# Resolución

In [8]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
calendario = proyecto.calendar(datos_proyecto['duration'])
duracion_minima = proyecto.duration(datos_proyecto['duration'])
pred = prelaciones.immediate_predecessor

## Apartado 1

In [9]:
display(Markdown(pregunta_1))

1. Para cada plan, compruebe actividad por actividad si respeta las prelaciones: una actividad no puede empezar antes de que terminen todas sus predecesoras. Liste las violaciones.  

In [10]:
def violaciones(inicio):
    fin = inicio + datos_proyecto['duration']
    filas = []
    for a in inicio.index:
        for p in sorted(pred[a]):
            if fin[p] > inicio[a]:
                filas.append({'actividad': a, 'empieza en': int(inicio[a]), 'predecesora': p, 'termina en': int(fin[p]), 'conflicto': int(fin[p] - inicio[a])})
    return pd.DataFrame(filas)
for plan in ('start_plan_1', 'start_plan_2'):
    v = violaciones(planes[plan])
    display(Markdown(f'**{plan}**: {len(v)} violación(es)'))
    if len(v): display(v.set_index('actividad'))

**start_plan_1**: 2 violación(es)

,empieza en,predecesora,termina en,conflicto
actividad,,,,
E,9,D,10,1
F,9,E,10,1


**start_plan_2**: 1 violación(es)

,empieza en,predecesora,termina en,conflicto
actividad,,,,
H,12,F,15,3


## Apartado 2

In [11]:
display(Markdown(pregunta_2))

2. Calcule la duración del proyecto que resulta de cada plan (fin de la última actividad) y compárela con la duración mínima del proyecto.  

In [12]:
fines = planes.add(datos_proyecto['duration'], axis='rows')
duraciones = fines.max().rename('fin del proyecto')
display(duraciones.to_frame().T)
Markdown(f'Duración mínima del proyecto: {duracion_minima}. Un plan con violaciones no es ejecutable tal cual: su duración aparente es engañosa.')

,start_plan_1,start_plan_2
fin del proyecto,17,17


Duración mínima del proyecto: 17. Un plan con violaciones no es ejecutable tal cual: su duración aparente es engañosa.

## Apartado 3

In [13]:
display(Markdown(pregunta_3))

3. Corrija cada plan con el mínimo cambio: adelante o retrase sólo las actividades necesarias (y sus sucesoras si hace falta) y recalcule la duración.  

Corrección mínima: se recorren las actividades en orden topológico y cada una empieza en el máximo entre la fecha propuesta y el fin de sus predecesoras (ya corregidas). Así sólo se mueven las actividades que lo necesitan y sus sucesoras.

In [14]:
G = nx.DiGraph(); G.add_nodes_from(datos_proyecto.index); G.add_edges_from((p, a) for a, ps in pred.items() for p in ps)
orden = list(nx.topological_sort(G))
def corregir(inicio):
    nuevo = inicio.copy()
    for a in orden:
        fin_pred = max([nuevo[p] + datos_proyecto.loc[p, 'duration'] for p in pred[a]], default=0)
        nuevo[a] = max(int(inicio[a]), int(fin_pred))
    return nuevo
corregidos = pd.DataFrame({plan: corregir(planes[plan]) for plan in planes.columns})
cambios = (corregidos != planes)
display(pd.concat({'propuesto': planes, 'corregido': corregidos}, axis=1))
fines_c = corregidos.add(datos_proyecto['duration'], axis='rows').max()
Markdown('Actividades movidas: ' + '; '.join(f'{plan}: {", ".join(cambios.index[cambios[plan]]) or "ninguna"}' for plan in planes.columns) + '. '
         + 'Duración tras corregir: ' + ', '.join(f'{plan}: {int(v)}' for plan, v in fines_c.items()) + '.')

propuesto                 corregido             
         start_plan_1 start_plan_2 start_plan_1 start_plan_2
activity                                                    
A                   0            0            0            0
B                   2            0            2            0
C                   3            0            3            0
D                   7            7            7            7
E                   9           12           10           12
F                   9           14           11           14
G                  10           10           10           10
H                  14           12           14           15

Actividades movidas: start_plan_1: E, F; start_plan_2: H. Duración tras corregir: start_plan_1: 17, start_plan_2: 18.

## Apartado 4

In [15]:
display(Markdown(pregunta_4))

4. ¿Qué plan es «más tardío» (deja menos margen) y qué riesgo supone? ¿Qué plan deja más holgura para absorber imprevistos?  

El margen de cada actividad en un plan es lo que puede retrasarse sin mover el fin *de ese plan*: su inicio más tardío (recalculado para la duración del plan corregido) menos el inicio programado.

In [16]:
margen = pd.DataFrame({plan: (calendario['inicio_mas_tardio'] + (fines_c[plan] - duracion_minima) - corregidos[plan]) for plan in corregidos.columns})
display(margen.rename(columns=lambda c: f'margen restante ({c})').T)
sin_margen = (margen <= 0).sum()
mas_tardio = sin_margen.idxmax()
Markdown(f'Actividades sin margen (empiezan en su inicio más tardío): ' + ', '.join(f'{p}: {int(v)}' for p, v in sin_margen.items()) + f'. '
         f'El plan más tardío es {mas_tardio}: en él casi cualquier retraso se traslada al fin del proyecto, porque las actividades han consumido su holgura antes de empezar. '
         'El otro plan conserva holgura para absorber imprevistos, a cambio de necesitar antes los recursos y de inmovilizar antes el dinero.')

activity,A,B,C,D,E,F,G,H
margen restante (start_plan_1),0,0,0,0,2,2,0,0
margen restante (start_plan_2),1,3,4,1,1,0,1,0


Actividades sin margen (empiezan en su inicio más tardío): start_plan_1: 6, start_plan_2: 2. El plan más tardío es start_plan_1: en él casi cualquier retraso se traslada al fin del proyecto, porque las actividades han consumido su holgura antes de empezar. El otro plan conserva holgura para absorber imprevistos, a cambio de necesitar antes los recursos y de inmovilizar antes el dinero.